# ResNet Pruning with Simulated Annealing

Load a pretrained torchvision ResNet-18 and use **simulated annealing** to find an
optimal per-layer sparsity allocation, then apply the resulting prune mask.

Compared to uniform global L1 pruning via `torch.nn.utils.prune`, SA explores the
combinatorial space of "how much to prune each layer" and accepts worse allocations
probabilistically to escape local minima.

In [ ]:
# Colab setup (skip if already cloned)
!apt-get install -y git-lfs > /dev/null 2>&1 && git lfs install
!git clone https://github.com/leozqi/resnet18_n2uq.git || true
%cd resnet18_n2uq

In [ ]:
import math
import copy
import torch
import torch.nn as nn
import torch.nn.utils.prune as prune
import torchvision.models as models
import matplotlib.pyplot as plt

torch.manual_seed(42)

CHECKPOINT = "pretrained_models/quantize_downsampling_false/resnet18-f37072fd(pytorch_model).pth"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 1. Load pretrained ResNet-18

In [ ]:
model = models.resnet18(weights=None)
checkpoint = torch.load(CHECKPOINT, map_location="cpu")
model.load_state_dict(checkpoint)
model = model.to(device)
model.eval()

for p in model.parameters():
    p.requires_grad_(False)

total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

## 2. Identify prunable layers

In [ ]:
def get_prunable_layers(module):
    """Return all Conv2d and Linear submodules."""
    layers = []
    for name, m in module.named_modules():
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            layers.append((name, m))
    return layers

prunable = get_prunable_layers(model)
n_layers = len(prunable)

print(f"Prunable layers: {n_layers}")
for i, (name, m) in enumerate(prunable):
    n = m.weight.nelement()
    print(f"  [{i:2d}] {name:40s}  {list(m.weight.shape)}  ({n:,} params)")

## 3. Calibration data (random proxy)

Replace with real ImageNet data for meaningful pruning.

In [ ]:
BATCH = 32

cal_x = torch.randn(BATCH, 3, 224, 224, device=device)
cal_y = torch.randint(0, 1000, (BATCH,), device=device)
loss_fn = nn.CrossEntropyLoss()

with torch.no_grad():
    bl_loss = loss_fn(model(cal_x), cal_y).item()
    print(f"Baseline loss: {bl_loss:.4f}")

## 4. SA helpers

Each “move” perturbs one layer’s sparsity fraction. The candidate is evaluated
by applying L1 pruning at those rates, computing CE loss, and adding a penalty
for deviating from the 50% global sparsity target.

In [ ]:
@torch.no_grad()
def snapshot_weights(model):
    """Return detached clone of every parameter."""
    return [p.detach().clone() for p in model.parameters()]

@torch.no_grad()
def load_weights(model, params):
    """Restore parameter values from a snapshot."""
    for p, s in zip(model.parameters(), params):
        p.copy_(s)


original_weights = snapshot_weights(model)


@torch.no_grad()
def evaluate_with_rates(sparsity_rates, cal_x, cal_y):
    """Apply per-layer L1 pruning at given rates, evaluate loss & acc, then undo."""
    for (_, m), rate in zip(prunable, sparsity_rates):
        if rate > 0:
            prune.l1_unstructured(m, name='weight', amount=float(rate))

    logits = model(cal_x)
    loss = loss_fn(logits, cal_y).item()
    acc = (logits.argmax(1) == cal_y).float().mean().item()

    # Undo
    for (_, m), _ in zip(prunable, sparsity_rates):
        if hasattr(m, 'weight_orig'):
            prune.remove(m, 'weight')

    return loss, acc


total_params_prunable = sum(m.weight.nelement() for _, m in prunable)


def energy(loss, sparsity_rates, target, lam):
    """Loss + weighted squared deviation from target sparsity."""
    remaining = sum(
        m.weight.nelement() * (1 - rate)
        for (_, m), rate in zip(prunable, sparsity_rates)
    )
    actual = 1.0 - remaining / total_params_prunable
    penalty = lam * (actual - target) ** 2
    return loss + penalty, actual


def perturb(rates, step):
    new = rates.copy()
    idx = torch.randint(0, len(rates), (1,)).item()
    delta = (torch.rand(1).item() * 2 - 1) * step
    new[idx] = float(max(0.0, min(0.95, new[idx] + delta)))
    return new


def accept(cand_e, curr_e, temp):
    delta = cand_e - curr_e
    if delta <= 0:
        return True
    return torch.rand(1).item() < math.exp(-delta / max(temp, 1e-12))

## 5. Run simulated annealing

In [ ]:
TARGET   = 0.50
ITERS    = 500
T0       = 0.05
ALPHA    = 0.99
T_MIN    = 1e-5
STEP     = 0.05
LAM      = 1.0

# Start with uniform sparsity
curr_rates = [TARGET] * n_layers
curr_loss, curr_acc = evaluate_with_rates(curr_rates, cal_x, cal_y)
curr_e, curr_sp = energy(curr_loss, curr_rates, TARGET, LAM)

best_rates = curr_rates.copy()
best_e = curr_e
best_loss = curr_loss

T = T0
history = []

for it in range(ITERS):
    if T < T_MIN:
        break

    cand_rates = perturb(curr_rates, STEP)
    cand_loss, cand_acc = evaluate_with_rates(cand_rates, cal_x, cal_y)
    cand_e, cand_sp = energy(cand_loss, cand_rates, TARGET, LAM)

    if accept(cand_e, curr_e, T):
        curr_rates = cand_rates
        curr_e = cand_e
        curr_loss = cand_loss
        curr_acc = cand_acc

        if curr_e < best_e:
            best_e = curr_e
            best_rates = curr_rates.copy()
            best_loss = curr_loss

    history.append((it, T, curr_e, best_e, curr_loss, curr_acc))
    T *= ALPHA

print(f"Done in {len(history)} iterations")
print(f"Best energy:  {best_e:.6f}")
print(f"Best loss:    {best_loss:.6f}")

## 6. Convergence plot

In [ ]:
iters    = [h[0] for h in history]
curr_es  = [h[2] for h in history]
best_es  = [h[3] for h in history]
losses   = [h[4] for h in history]
temps    = [h[1] for h in history]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(iters, curr_es, alpha=0.4, label='current')
axes[0].plot(iters, best_es, label='best', linewidth=2)
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Energy')
axes[0].set_title('SA Energy')
axes[0].legend()

axes[1].plot(iters, losses)
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('CE Loss')
axes[1].set_title('Current Loss')

axes[2].plot(iters, temps)
axes[2].set_xlabel('Iteration')
axes[2].set_ylabel('Temperature')
axes[2].set_title('Cooling Schedule')
axes[2].set_yscale('log')

plt.tight_layout()
plt.show()

## 7. Found sparsity allocation

In [ ]:
layer_labels = [name for name, _ in prunable]

fig, ax = plt.subplots(figsize=(10, 8))
y = range(n_layers)
ax.barh(y, [r * 100 for r in best_rates])
ax.set_yticks(y)
ax.set_yticklabels(layer_labels, fontsize=8)
ax.set_xlabel('Sparsity (%)')
ax.set_title(f"SA-optimal per-layer sparsity (target {TARGET:.0%})")
ax.axvline(x=TARGET * 100, color='r', linestyle='--', label=f'target {TARGET:.0%}')
ax.legend()
plt.tight_layout()
plt.show()

avg = sum(best_rates) / len(best_rates)
print(f"Average sparsity: {avg:.1%}")

## 8. Apply best mask permanently

In [ ]:
# Reload clean weights
load_weights(model, original_weights)

# Apply SA-optimized per-layer pruning permanently
for (_, m), rate in zip(prunable, best_rates):
    if rate > 0:
        prune.l1_unstructured(m, name='weight', amount=float(rate))
        prune.remove(m, 'weight')

# Verify
with torch.no_grad():
    final_loss = loss_fn(model(cal_x), cal_y).item()
print(f"Final loss (SA-pruned): {final_loss:.6f}")

# Count total sparsity
total_nz = sum(torch.count_nonzero(p).item() for p in model.parameters())
print(f"Sparsity: {1 - total_nz/total_params:.1%}")
print(f"Nonzero: {total_nz:,} / {total_params:,}")

save_path = "resnet18_sa_pruned.pth"
torch.save(model.state_dict(), save_path)
print(f"Saved to {save_path}")

## 9. Compare: uniform pruning vs SA-optimized

In [ ]:
# Rebuild clean model for uniform-pruning baseline
model_uniform = models.resnet18(weights=None)
model_uniform.load_state_dict(checkpoint)
model_uniform = model_uniform.to(device)
model_uniform.eval()
for p in model_uniform.parameters():
    p.requires_grad_(False)

prunable_u = get_prunable_layers(model_uniform)

# Apply uniform 50% global pruning
params_u = [(m, 'weight') for _, m in prunable_u]
prune.global_unstructured(params_u, pruning_method=prune.L1Unstructured, amount=TARGET)
for _, m in prunable_u:
    prune.remove(m, 'weight')

with torch.no_grad():
    uniform_loss = loss_fn(model_uniform(cal_x), cal_y).item()

names = ['Baseline', 'Uniform 50%', 'SA-optimized']
losses = [bl_loss, uniform_loss, final_loss]

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(names, losses, color=['#4CAF50', '#2196F3', '#FF9800'])
ax.set_ylabel('Cross-Entropy Loss')
ax.set_title('Pruning method comparison')
for i, v in enumerate(losses):
    ax.text(i, v, f'{v:.4f}', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()